## 18. Model Validation

### Objective

This notebook validates the saved traditional machine-learning and deep-learning model artifacts.

The validation includes:

1. Checking model files and supporting artifacts.
2. Loading traditional ML models and verifying their input requirements.
3. Loading deep-learning checkpoints using their exact architectures.
4. Checking prediction shapes, class mappings, and numerical validity.
5. Comparing regenerated predictions with previously saved predictions where applicable.
6. Recording validation results without retraining any models.

**Important:** This notebook performs validation only. It does not modify earlier notebooks or overwrite the saved model artifacts.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

MODELS_DIR = PROJECT_ROOT / "models"

print("Models directory exists:", MODELS_DIR.exists())
print("Models directory:", MODELS_DIR)

if MODELS_DIR.exists():
    for path in sorted(MODELS_DIR.rglob("*")):
        if path.is_file():
            print(
                f"{path.relative_to(PROJECT_ROOT)} "
                f"| {path.stat().st_size / (1024**2):.2f} MB"
            )
        elif path.is_dir():
            print(f"[DIR] {path.relative_to(PROJECT_ROOT)}")

Models directory exists: True
Models directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/models
models/.DS_Store | 0.01 MB
[DIR] models/deep_learning
[DIR] models/deep_learning/cnn_lstm
[DIR] models/deep_learning/cnn_lstm/.ipynb_checkpoints
models/deep_learning/cnn_lstm/best_cnn_lstm.pt | 0.36 MB
models/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv | 0.20 MB
models/deep_learning/cnn_lstm/cnn_lstm_training_config.json | 0.00 MB
models/deep_learning/cnn_lstm/cnn_lstm_training_history.csv | 0.00 MB
models/deep_learning/cnn_lstm/logmel_train_normalization.csv | 0.00 MB
[DIR] models/deep_learning/lightweight_mel_cnn
models/deep_learning/lightweight_mel_cnn/best_lightweight_mel_cnn.pt | 0.30 MB
models/deep_learning/lightweight_mel_cnn/cnn_test_predictions.csv | 0.21 MB
models/deep_learning/lightweight_mel_cnn/cnn_training_config.json | 0.00 MB
models/deep_learning/lightweight_mel_cnn/cnn_training_history.csv | 0.00 MB
models/deep_learning/lightweight_mel_cnn/logmel_train_nor

In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib
import torch

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

MODELS_DIR = PROJECT_ROOT / "models"

TRADITIONAL_DIR = MODELS_DIR / "traditional_ml"
DL_DIR = MODELS_DIR / "deep_learning"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Models directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/models
Device: cpu
PyTorch version: 2.14.1
NumPy version: 2.5.3
Pandas version: 3.0.6


In [3]:
expected_artifacts = {
    "Logistic Regression": (
        TRADITIONAL_DIR / "logistic_regression.joblib"
    ),
    "SVM": (
        TRADITIONAL_DIR / "svm.joblib"
    ),
    "Random Forest": (
        TRADITIONAL_DIR / "random_forest.joblib"
    ),
    "Scaler": TRADITIONAL_DIR / "scaler.joblib",
    "Label Encoder": TRADITIONAL_DIR / "label_encoder.joblib",
    "Selected Features": TRADITIONAL_DIR / "selected_features.json",
    "Classical Metadata": TRADITIONAL_DIR / "classical_ml_metadata.json",
    "Lightweight Mel CNN": (
        DL_DIR / "lightweight_mel_cnn" /
        "best_lightweight_mel_cnn.pt"
    ),
    "Multi-feature CNN": (
        DL_DIR / "multifeature_cnn" /
        "best_multifeature_cnn.pt"
    ),
    "Regularized Multi-feature CNN": (
        DL_DIR / "regularized_multifeature_cnn" /
        "best_regularized_multifeature_cnn.pt"
    ),
    "CNN-LSTM": (
        DL_DIR / "cnn_lstm" / "best_cnn_lstm.pt"
    ),
}

artifact_rows = []

for name, path in expected_artifacts.items():
    artifact_rows.append({
        "Artifact": name,
        "Exists": path.is_file(),
        "Size (MB)": (
            round(path.stat().st_size / (1024 ** 2), 3)
            if path.is_file() else np.nan
        ),
        "Path": str(path),
    })

artifact_df = pd.DataFrame(artifact_rows)

display(artifact_df)

missing_artifacts = artifact_df.loc[
    ~artifact_df["Exists"], "Artifact"
].tolist()

if missing_artifacts:
    print("Missing artifacts:", missing_artifacts)
else:
    print("All expected model and supporting artifacts exist.")

,Artifact,Exists,Size (MB),Path
0,Logistic Regression,True,0.003,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,SVM,True,0.867,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,Random Forest,True,21.922,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
3,Scaler,True,0.002,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
4,Label Encoder,True,0.001,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
5,Selected Features,True,0.001,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
6,Classical Metadata,True,0.003,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
7,Lightweight Mel CNN,True,0.297,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
8,Multi-feature CNN,True,0.443,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
9,Regularized Multi-feature CNN,True,0.444,/Users/vs/Sem 5/Respiratory-Disease-Classifica...


All expected model and supporting artifacts exist.


In [4]:
METADATA_PATH = (
    TRADITIONAL_DIR / "classical_ml_metadata.json"
)

FEATURES_PATH = (
    TRADITIONAL_DIR / "selected_features.json"
)

with open(METADATA_PATH, "r") as f:
    classical_metadata = json.load(f)

with open(FEATURES_PATH, "r") as f:
    selected_features_data = json.load(f)

print("CLASSICAL ML METADATA")
print(json.dumps(classical_metadata, indent=2))

print("\nSELECTED FEATURES")
print(json.dumps(selected_features_data, indent=2))

CLASSICAL ML METADATA
{
  "project": "Respiratory Disease Classification",
  "source_notebook": "notebooks/09_classical_ml.ipynb",
  "training_rows": 4142,
  "test_rows": 2756,
  "feature_count": 33,
  "selected_feature_names": [
    "spectral_entropy",
    "mfcc_11_mean",
    "zero_crossing_rate",
    "dominant_frequency",
    "mfcc_13_mean",
    "wavelet_entropy",
    "mfcc_1_mean",
    "spectral_bandwidth",
    "spectral_rolloff",
    "mfcc_5_mean",
    "mfcc_3_mean",
    "mfcc_8_mean",
    "peak_to_peak",
    "mfcc_9_mean",
    "mfcc_12_mean",
    "mfcc_7_mean",
    "mfcc_2_mean",
    "energy",
    "rms",
    "mfcc_6_mean",
    "band_energy_500_1000",
    "spectral_flatness",
    "band_energy_0_500",
    "mfcc_4_mean",
    "band_energy_1000_1500",
    "min",
    "mfcc_10_mean",
    "skewness",
    "mean",
    "kurtosis",
    "median",
    "band_energy_1500_2000",
    "max"
  ],
  "class_names": [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
  

In [5]:
# Load saved traditional ML artifacts

logistic_model = joblib.load(
    TRADITIONAL_DIR / "logistic_regression.joblib"
)

svm_model = joblib.load(
    TRADITIONAL_DIR / "svm.joblib"
)

random_forest_model = joblib.load(
    TRADITIONAL_DIR / "random_forest.joblib"
)

scaler = joblib.load(
    TRADITIONAL_DIR / "scaler.joblib"
)

label_encoder = joblib.load(
    TRADITIONAL_DIR / "label_encoder.joblib"
)

print("Traditional ML artifacts loaded successfully.")
print("Logistic Regression:", type(logistic_model).__name__)
print("SVM:", type(svm_model).__name__)
print("Random Forest:", type(random_forest_model).__name__)
print("Scaler:", type(scaler).__name__)
print("Label Encoder:", type(label_encoder).__name__)

Traditional ML artifacts loaded successfully.
Logistic Regression: LogisticRegression
SVM: SVC
Random Forest: RandomForestClassifier
Scaler: StandardScaler
Label Encoder: LabelEncoder


In [6]:
# Validate expected feature count and class labels

expected_feature_count = len(selected_features_data)

print("Selected feature count:", expected_feature_count)
print("Metadata feature count:", classical_metadata["feature_count"])

print("\nModel input dimensions:")
print("Logistic Regression:", logistic_model.n_features_in_)
print("SVM:", svm_model.n_features_in_)
print("Random Forest:", random_forest_model.n_features_in_)

print("\nLabel encoder classes:")
print(label_encoder.classes_)

print("\nMetadata classes:")
print(classical_metadata["class_names"])

assert expected_feature_count == classical_metadata["feature_count"]

assert logistic_model.n_features_in_ == expected_feature_count
assert svm_model.n_features_in_ == expected_feature_count
assert random_forest_model.n_features_in_ == expected_feature_count

print("\nAll feature-dimension checks passed.")

Selected feature count: 33
Metadata feature count: 33

Model input dimensions:
Logistic Regression: 33
SVM: 33
Random Forest: 33

Label encoder classes:
['Asthma' 'Bronchiectasis' 'Bronchiolitis' 'COPD' 'Healthy' 'LRTI'
 'Pneumonia' 'URTI']

Metadata classes:
['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']

All feature-dimension checks passed.


In [7]:
print("Scaler type:", type(scaler).__name__)

if hasattr(scaler, "n_features_in_"):
    print("Scaler expected features:", scaler.n_features_in_)

if hasattr(scaler, "mean_"):
    print("Scaler mean shape:", scaler.mean_.shape)

if hasattr(scaler, "scale_"):
    print("Scaler scale shape:", scaler.scale_.shape)

assert scaler.n_features_in_ == expected_feature_count

print("\nScaler feature-dimension check passed.")

Scaler type: StandardScaler
Scaler expected features: 33
Scaler mean shape: (33,)
Scaler scale shape: (33,)

Scaler feature-dimension check passed.


In [8]:
CLASSICAL_RESULTS_DIR = (
    PROJECT_ROOT / "data" / "processed" / "classical_ml"
)

print("Files in classical ML results directory:")

for path in sorted(CLASSICAL_RESULTS_DIR.iterdir()):
    print(path.name)

Files in classical ML results directory:
.ipynb_checkpoints
classical_ml_model_comparison.csv
classical_ml_per_disease_results.csv
classical_ml_performance_comparison.png
classical_ml_predictions.csv
classical_ml_selected_features.csv
final_classical_ml_comparison.csv
random_forest_feature_group_importance.csv
random_forest_feature_importance.csv


In [9]:
CLASSICAL_PREDICTIONS_PATH = (
    CLASSICAL_RESULTS_DIR / "classical_ml_predictions.csv"
)

classical_predictions_df = pd.read_csv(
    CLASSICAL_PREDICTIONS_PATH
)

print("Shape:", classical_predictions_df.shape)
print("Columns:", classical_predictions_df.columns.tolist())

display(classical_predictions_df.head())

Shape: (2756, 7)
Columns: ['patient_id', 'recording_id', 'chest_location', 'true_diagnosis', 'logistic_regression_prediction', 'svm_prediction', 'random_forest_prediction']


,patient_id,recording_id,chest_location,true_diagnosis,logistic_regression_prediction,svm_prediction,random_forest_prediction
0,101,1b1,Al,URTI,Healthy,Pneumonia,URTI
1,101,1b1,Al,URTI,URTI,Healthy,URTI
2,101,1b1,Al,URTI,Healthy,Healthy,Healthy
3,101,1b1,Al,URTI,URTI,URTI,URTI
4,101,1b1,Al,URTI,Pneumonia,Healthy,URTI


In [10]:
FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

RECORDINGS_PATH = (
    PROJECT_ROOT / "data" / "interim" / "recordings.csv"
)

features_df = pd.read_csv(FEATURES_PATH)
recordings_df = pd.read_csv(RECORDINGS_PATH)

print("Feature data shape:", features_df.shape)
print("Recording metadata shape:", recordings_df.shape)

print("\nFeature columns:")
print(features_df.columns.tolist())

print("\nRecording columns:")
print(recordings_df.columns.tolist())

Feature data shape: (6898, 52)
Recording metadata shape: (920, 7)

Feature columns:
['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'cycle_number', 'start_time', 'end_time', 'sound_label', 'crackles', 'wheezes', 'diagnosis', 'processed_path', 'mean', 'std', 'variance', 'rms', 'max', 'min', 'peak_to_peak', 'median', 'skewness', 'kurtosis', 'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'spectral_energy', 'dominant_frequency', 'spectral_entropy', 'band_energy_0_500', 'band_energy_500_1000', 'band_energy_1000_1500', 'band_energy_1500_2000', 'mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_9_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'wavelet_energy', 'wavelet_entropy', 'wavelet_std']

Recording columns:
['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 

In [11]:
# Attach official split information to the cycle features

merge_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
]

features_with_split = features_df.merge(
    recordings_df[
        merge_keys + ["official_split"]
    ],
    on=merge_keys,
    how="left",
    validate="many_to_one",
)

print("Rows after merge:", len(features_with_split))
print("Missing split labels:", features_with_split["official_split"].isna().sum())

print("\nSplit counts:")
print(features_with_split["official_split"].value_counts(dropna=False))

Rows after merge: 6898
Missing split labels: 0

Split counts:
official_split
train    4142
test     2756
Name: count, dtype: int64


In [12]:
# Use the official test split and exact saved feature order

test_features_df = features_with_split.loc[
    features_with_split["official_split"].astype(str).str.lower().eq("test")
].copy()

X_test_classical = test_features_df[
    selected_features_data
].copy()

y_test_classical = label_encoder.transform(
    test_features_df["diagnosis"]
) if "diagnosis" in test_features_df.columns else None

print("Test feature matrix:", X_test_classical.shape)
print("Expected feature count:", expected_feature_count)

assert X_test_classical.shape[1] == expected_feature_count
assert np.isfinite(X_test_classical.to_numpy(dtype=float)).all()

print("Test feature matrix is finite and has the expected dimensions.")

Test feature matrix: (2756, 33)
Expected feature count: 33
Test feature matrix is finite and has the expected dimensions.


In [13]:
# Prepare scaled and unscaled inputs

X_test_scaled = scaler.transform(X_test_classical)

# Run inference without retraining

lr_predictions = logistic_model.predict(X_test_scaled)
svm_predictions = svm_model.predict(X_test_scaled)
rf_predictions = random_forest_model.predict(X_test_classical)

print("Prediction counts:")
print("Logistic Regression:", len(lr_predictions))
print("SVM:", len(svm_predictions))
print("Random Forest:", len(rf_predictions))

print("\nFirst 10 encoded predictions:")
print("Logistic Regression:", lr_predictions[:10])
print("SVM:", svm_predictions[:10])
print("Random Forest:", rf_predictions[:10])

assert len(lr_predictions) == len(X_test_classical)
assert len(svm_predictions) == len(X_test_classical)
assert len(rf_predictions) == len(X_test_classical)

print("\nAll three models generated predictions successfully.")

Prediction counts:
Logistic Regression: 2756
SVM: 2756
Random Forest: 2756

First 10 encoded predictions:
Logistic Regression: [4 7 4 7 6 7 4 4 4 4]
SVM: [6 4 4 7 4 7 4 4 4 4]
Random Forest: [7 7 4 7 7 7 4 7 3 7]

All three models generated predictions successfully.


In [14]:
print("Saved prediction columns:")
print(classical_predictions_df.columns.tolist())

display(classical_predictions_df.head())

Saved prediction columns:
['patient_id', 'recording_id', 'chest_location', 'true_diagnosis', 'logistic_regression_prediction', 'svm_prediction', 'random_forest_prediction']


,patient_id,recording_id,chest_location,true_diagnosis,logistic_regression_prediction,svm_prediction,random_forest_prediction
0,101,1b1,Al,URTI,Healthy,Pneumonia,URTI
1,101,1b1,Al,URTI,URTI,Healthy,URTI
2,101,1b1,Al,URTI,Healthy,Healthy,Healthy
3,101,1b1,Al,URTI,URTI,URTI,URTI
4,101,1b1,Al,URTI,Pneumonia,Healthy,URTI


In [16]:
# Compare regenerated predictions against saved disease-name predictions

prediction_mapping = {
    "Logistic Regression": (
        lr_predictions,
        "logistic_regression_prediction",
    ),
    "SVM": (
        svm_predictions,
        "svm_prediction",
    ),
    "Random Forest": (
        rf_predictions,
        "random_forest_prediction",
    ),
}

for model_name, (new_predictions, saved_column) in prediction_mapping.items():

    if saved_column not in classical_predictions_df.columns:
        print(f"{model_name}: column '{saved_column}' not found.")
        continue

    saved_predictions = (
        classical_predictions_df[saved_column]
        .astype(str)
        .to_numpy()
    )

    # Convert encoded predictions into disease names
    new_prediction_names = label_encoder.inverse_transform(
        np.asarray(new_predictions, dtype=int)
    )

    if len(saved_predictions) != len(new_prediction_names):
        print(f"{model_name}: row count mismatch.")
        continue

    agreement = np.mean(
        saved_predictions == new_prediction_names
    )

    print(f"{model_name}:")
    print(f"  Samples compared: {len(saved_predictions)}")
    print(f"  Exact prediction agreement: {agreement:.4%}")

Logistic Regression:
  Samples compared: 2756
  Exact prediction agreement: 100.0000%
SVM:
  Samples compared: 2756
  Exact prediction agreement: 100.0000%
Random Forest:
  Samples compared: 2756
  Exact prediction agreement: 100.0000%


## Deep-learning validation
Inspect each model's configuration
We should inspect the saved configurations before recreating architectures or loading checkpoints. Different models may require different input features and normalization.

In [17]:
DL_MODEL_DIRS = {
    "Lightweight Mel CNN": (
        DL_DIR / "lightweight_mel_cnn"
    ),
    "Multi-feature CNN": (
        DL_DIR / "multifeature_cnn"
    ),
    "Regularized Multi-feature CNN": (
        DL_DIR / "regularized_multifeature_cnn"
    ),
    "CNN-LSTM": (
        DL_DIR / "cnn_lstm"
    ),
}

for model_name, model_dir in DL_MODEL_DIRS.items():
    print("\n" + "=" * 70)
    print(model_name)

    config_files = list(model_dir.glob("*config*.json"))

    if not config_files:
        print("No configuration JSON found.")
        continue

    for config_path in config_files:
        print("\nConfiguration file:", config_path.name)

        with open(config_path, "r") as f:
            config = json.load(f)

        print(json.dumps(config, indent=2))


Lightweight Mel CNN

Configuration file: cnn_training_config.json
{
  "batch_size": 32,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "lr_reduction_factor": 0.5,
  "lr_scheduler_patience": 3,
  "minimum_learning_rate": 1e-06,
  "optimizer": "Adam",
  "loss": "Weighted CrossEntropyLoss",
  "augmentation": [
    "Gaussian noise",
    "random time shift"
  ],
  "augmentation_training_only": true,
  "validation_monitor": "loss"
}

Multi-feature CNN

Configuration file: multifeature_cnn_training_config.json
{
  "batch_size": 32,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "scheduler": "ReduceLROnPlateau",
  "scheduler_factor": 0.5,
  "scheduler_patience": 3,
  "scheduler_min_lr": 1e-06,
  "loss": "weighted_cross_entropy",
  "optimizer": "Adam",
  "augmentation": [
    "random_time_shift",
    "gaussian_noise"
  ],
  "feature_inputs": [
    "Log-Mel",
    "MFCC",
    "Ch

In [18]:
checkpoint_paths = {
    "Lightweight Mel CNN": (
        DL_MODEL_DIRS["Lightweight Mel CNN"]
        / "best_lightweight_mel_cnn.pt"
    ),
    "Multi-feature CNN": (
        DL_MODEL_DIRS["Multi-feature CNN"]
        / "best_multifeature_cnn.pt"
    ),
    "Regularized Multi-feature CNN": (
        DL_MODEL_DIRS["Regularized Multi-feature CNN"]
        / "best_regularized_multifeature_cnn.pt"
    ),
    "CNN-LSTM": (
        DL_MODEL_DIRS["CNN-LSTM"]
        / "best_cnn_lstm.pt"
    ),
}

for model_name, checkpoint_path in checkpoint_paths.items():
    print("\n" + "=" * 70)
    print(model_name)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys()))

        state_dict = checkpoint.get(
            "model_state_dict",
            checkpoint
        )

        if isinstance(state_dict, dict):
            print("Number of parameter entries:", len(state_dict))

            for key, value in list(state_dict.items())[:8]:
                if torch.is_tensor(value):
                    print(f"  {key}: {tuple(value.shape)}")
                else:
                    print(f"  {key}: {type(value).__name__}")
    else:
        print("Unexpected checkpoint type:", type(checkpoint))


Lightweight Mel CNN
Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_accuracy', 'class_mapping', 'config']
Number of parameter entries: 23
  features.0.weight: (16, 1, 3, 3)
  features.0.bias: (16,)
  features.1.weight: (16,)
  features.1.bias: (16,)
  features.1.running_mean: (16,)
  features.1.running_var: (16,)
  features.1.num_batches_tracked: ()
  features.4.weight: (32, 16, 3, 3)

Multi-feature CNN
Top-level keys: ['logmel_branch.features.0.weight', 'logmel_branch.features.0.bias', 'logmel_branch.features.1.weight', 'logmel_branch.features.1.bias', 'logmel_branch.features.1.running_mean', 'logmel_branch.features.1.running_var', 'logmel_branch.features.1.num_batches_tracked', 'logmel_branch.features.4.weight', 'logmel_branch.features.4.bias', 'logmel_branch.features.5.weight', 'logmel_branch.features.5.bias', 'logmel_branch.features.5.running_mean', 'logmel_branch.features.5.running_var', 'logmel_branch.features.5.num_batches_tracked', 'logme

In [19]:
checkpoint_validation_rows = []

for model_name, checkpoint_path in checkpoint_paths.items():

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        class_mapping = checkpoint.get("class_mapping", None)
    else:
        state_dict = checkpoint
        class_mapping = None

    tensor_count = 0
    non_finite_tensors = []
    total_parameters = 0

    for key, value in state_dict.items():
        if torch.is_tensor(value):
            tensor_count += 1
            total_parameters += value.numel()

            if not torch.isfinite(value).all():
                non_finite_tensors.append(key)

    # Identify the classifier's output dimension from its weight matrix
    output_dimensions = [
        tuple(value.shape)
        for key, value in state_dict.items()
        if "classifier" in key.lower()
        and key.endswith("weight")
        and torch.is_tensor(value)
        and value.ndim == 2
    ]

    checkpoint_validation_rows.append({
        "Model": model_name,
        "Tensor entries": tensor_count,
        "Parameter values": total_parameters,
        "Output weight shapes": str(output_dimensions),
        "Non-finite tensors": len(non_finite_tensors),
        "Class mapping available": class_mapping is not None,
    })

    if non_finite_tensors:
        print(
            f"WARNING — {model_name} has non-finite tensors:",
            non_finite_tensors
        )

checkpoint_validation_df = pd.DataFrame(
    checkpoint_validation_rows
)

display(checkpoint_validation_df)

,Model,Tensor entries,Parameter values,Output weight shapes,Non-finite tensors,Class mapping available
0,Lightweight Mel CNN,23,24267,"[(8, 64)]",0,True
1,Multi-feature CNN,73,109457,"[(128, 192), (8, 128)]",0,False
2,Regularized Multi-feature CNN,73,109457,"[(128, 192), (8, 128)]",0,False
3,CNN-LSTM,31,90827,"[(8, 64)]",0,False


In [20]:
for model_name, model_dir in DL_MODEL_DIRS.items():

    print("\n" + "=" * 70)
    print(model_name)

    normalization_files = list(
        model_dir.glob("*normalization*.csv")
    )

    if not normalization_files:
        print("No normalization CSV found in this model directory.")
        continue

    for norm_path in normalization_files:
        norm_df = pd.read_csv(norm_path)

        print("\nFile:", norm_path.name)
        print("Shape:", norm_df.shape)
        print("Columns:", norm_df.columns.tolist())
        display(norm_df.head())


Lightweight Mel CNN

File: logmel_train_normalization.csv
Shape: (2, 2)
Columns: ['statistic', 'value']


,statistic,value
0,mean,-66.740814
1,std,17.337090



Multi-feature CNN

File: multifeature_train_normalization.csv
Shape: (3, 4)
Columns: ['feature', 'count', 'mean', 'std']


,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059



Regularized Multi-feature CNN
No normalization CSV found in this model directory.

CNN-LSTM

File: logmel_train_normalization.csv
Shape: (2, 2)
Columns: ['statistic', 'value']


,statistic,value
0,mean,-66.740814
1,std,17.337090


In [21]:
prediction_files = {
    "Lightweight Mel CNN": (
        DL_MODEL_DIRS["Lightweight Mel CNN"]
        / "cnn_test_predictions.csv"
    ),
    "Multi-feature CNN": (
        DL_MODEL_DIRS["Multi-feature CNN"]
        / "multifeature_cnn_test_predictions.csv"
    ),
    "Regularized Multi-feature CNN": (
        DL_MODEL_DIRS["Regularized Multi-feature CNN"]
        / "test_predictions.csv"
    ),
    "CNN-LSTM": (
        DL_MODEL_DIRS["CNN-LSTM"]
        / "cnn_lstm_test_predictions.csv"
    ),
}

for model_name, prediction_path in prediction_files.items():

    print("\n" + "=" * 70)
    print(model_name)

    if not prediction_path.exists():
        print("Prediction file missing:", prediction_path)
        continue

    prediction_df = pd.read_csv(prediction_path)

    print("Rows:", len(prediction_df))
    print("Columns:", prediction_df.columns.tolist())
    display(prediction_df.head(3))


Lightweight Mel CNN
Rows: 1561
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'true_label', 'predicted_label', 'true_disease', 'predicted_disease']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split,true_label,predicted_label,true_disease,predicted_disease
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,7,5,URTI,LRTI
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,7,2,URTI,Bronchiolitis
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,7,2,URTI,Bronchiolitis



Multi-feature CNN
Rows: 1561
Columns: ['true_label', 'predicted_label', 'prob_Asthma', 'prob_Bronchiectasis', 'prob_Bronchiolitis', 'prob_COPD', 'prob_Healthy', 'prob_LRTI', 'prob_Pneumonia', 'prob_URTI']


,true_label,predicted_label,prob_Asthma,prob_Bronchiectasis,prob_Bronchiolitis,prob_COPD,prob_Healthy,prob_LRTI,prob_Pneumonia,prob_URTI
0,7,3,0.000028,0.000314,0.002751,0.984093,0.007683,0.001315,0.002192,0.001625
1,7,3,0.000061,0.000236,0.001470,0.984799,0.007738,0.000661,0.003757,0.001276
2,7,3,0.000123,0.000345,0.003772,0.969177,0.015415,0.001855,0.006571,0.002743



Regularized Multi-feature CNN
Rows: 1561
Columns: ['true_label', 'predicted_label', 'prob_Asthma', 'prob_Bronchiectasis', 'prob_Bronchiolitis', 'prob_COPD', 'prob_Healthy', 'prob_LRTI', 'prob_Pneumonia', 'prob_URTI']


,true_label,predicted_label,prob_Asthma,prob_Bronchiectasis,prob_Bronchiolitis,prob_COPD,prob_Healthy,prob_LRTI,prob_Pneumonia,prob_URTI
0,7,3,0.006300,0.136707,0.065136,0.602875,0.039925,0.019768,0.097903,0.031387
1,7,3,0.012703,0.130662,0.073895,0.542733,0.053770,0.029238,0.115558,0.041441
2,7,3,0.010876,0.132233,0.073360,0.554080,0.050549,0.027199,0.112058,0.039646



CNN-LSTM
Rows: 1561
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'predicted_disease_label', 'predicted_diagnosis']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split,predicted_disease_label,predicted_diagnosis
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD


In [22]:
# Inspect normalization-related checkpoint/config metadata

for model_name in [
    "Lightweight Mel CNN",
    "Multi-feature CNN",
    "Regularized Multi-feature CNN",
    "CNN-LSTM",
]:
    print("\n" + "=" * 70)
    print(model_name)

    model_dir = DL_MODEL_DIRS[model_name]
    config_files = list(model_dir.glob("*config*.json"))

    for config_path in config_files:
        with open(config_path, "r") as f:
            config = json.load(f)

        print("Config:", config_path.name)
        print("Feature inputs:", config.get("feature_inputs"))
        print("Features:", config.get("features"))
        print("Normalization:", config.get("normalization"))
        print("Normalization file:", config.get("normalization_file"))

    checkpoint = torch.load(
        checkpoint_paths[model_name],
        map_location="cpu",
        weights_only=True,
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        print("Checkpoint config:")
        print(json.dumps(checkpoint.get("config", {}), indent=2))
        print("Checkpoint class mapping:")
        print(checkpoint.get("class_mapping"))
    else:
        print("Raw state_dict checkpoint; normalization metadata not embedded.")


Lightweight Mel CNN
Config: cnn_training_config.json
Feature inputs: None
Features: None
Normalization: None
Normalization file: None
Checkpoint config:
{
  "batch_size": 32,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "lr_reduction_factor": 0.5,
  "lr_scheduler_patience": 3,
  "minimum_learning_rate": 1e-06,
  "optimizer": "Adam",
  "loss": "Weighted CrossEntropyLoss",
  "augmentation": [
    "Gaussian noise",
    "random time shift"
  ],
  "augmentation_training_only": true,
  "validation_monitor": "loss"
}
Checkpoint class mapping:
[{'disease_label': 0, 'diagnosis': 'Asthma'}, {'disease_label': 1, 'diagnosis': 'Bronchiectasis'}, {'disease_label': 2, 'diagnosis': 'Bronchiolitis'}, {'disease_label': 3, 'diagnosis': 'COPD'}, {'disease_label': 4, 'diagnosis': 'Healthy'}, {'disease_label': 5, 'diagnosis': 'LRTI'}, {'disease_label': 6, 'diagnosis': 'Pneumonia'}, {'disease_label': 7, 'diagnosis': 'URTI'}]

Multi-feature CNN
Con

In [23]:
# Validate saved deep-learning prediction tables

class_names = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

dl_prediction_validation = []

for model_name, prediction_path in prediction_files.items():
    df = pd.read_csv(prediction_path)

    row = {
        "Model": model_name,
        "Rows": len(df),
        "True labels valid": (
            "true_label" in df.columns
            and df["true_label"].dropna().between(0, 7).all()
        ),
    }

    if "predicted_label" in df.columns:
        pred_col = "predicted_label"
    elif "predicted_disease_label" in df.columns:
        pred_col = "predicted_disease_label"
    else:
        pred_col = None

    row["Prediction column"] = pred_col
    row["Predicted labels valid"] = (
        pred_col is not None
        and df[pred_col].dropna().between(0, 7).all()
    )

    prob_cols = [f"prob_{name}" for name in class_names]

    if all(c in df.columns for c in prob_cols):
        probs = df[prob_cols].to_numpy(dtype=float)

        row["Probabilities finite"] = np.isfinite(probs).all()
        row["Probabilities in range"] = (
            (probs >= 0).all() and (probs <= 1).all()
        )
        row["Max probability-sum error"] = float(
            np.abs(probs.sum(axis=1) - 1).max()
        )
    else:
        row["Probabilities finite"] = "N/A"
        row["Probabilities in range"] = "N/A"
        row["Max probability-sum error"] = "N/A"

    dl_prediction_validation.append(row)

dl_prediction_validation_df = pd.DataFrame(
    dl_prediction_validation
)

display(dl_prediction_validation_df)

,Model,Rows,True labels valid,Prediction column,Predicted labels valid,Probabilities finite,Probabilities in range,Max probability-sum error
0,Lightweight Mel CNN,1561,True,predicted_label,True,N/A,N/A,N/A
1,Multi-feature CNN,1561,True,predicted_label,True,True,True,0.0
2,Regularized Multi-feature CNN,1561,True,predicted_label,True,True,True,0.0
3,CNN-LSTM,1561,False,predicted_disease_label,True,N/A,N/A,N/A


In [24]:

cnn_lstm_predictions_path = (
    DL_DIR / "cnn_lstm" / "cnn_lstm_test_predictions.csv"
)

cnn_lstm_df = pd.read_csv(cnn_lstm_predictions_path)

print("Shape:", cnn_lstm_df.shape)
print("\nColumns:")
print(cnn_lstm_df.columns.tolist())

print("\nDiagnosis values:")
print(cnn_lstm_df["diagnosis"].value_counts(dropna=False))

print("\nPredicted diagnosis values:")
print(cnn_lstm_df["predicted_diagnosis"].value_counts(dropna=False))

print("\nMissing values:")
print(
    cnn_lstm_df[
        ["diagnosis", "predicted_disease_label", "predicted_diagnosis"]
    ].isna().sum()
)

print("\nFirst 10 rows:")
display(
    cnn_lstm_df[
        [
            "patient_id",
            "recording_id",
            "diagnosis",
            "predicted_disease_label",
            "predicted_diagnosis",
        ]
    ].head(10)
)


Shape: (1561, 10)

Columns:
['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'predicted_disease_label', 'predicted_diagnosis']

Diagnosis values:
diagnosis
COPD              1337
Healthy             84
Pneumonia           52
URTI                49
Bronchiectasis      20
Bronchiolitis       19
Name: count, dtype: int64

Predicted diagnosis values:
predicted_diagnosis
COPD         1411
Pneumonia     115
Asthma         35
Name: count, dtype: int64

Missing values:
diagnosis                  0
predicted_disease_label    0
predicted_diagnosis        0
dtype: int64

First 10 rows:


,patient_id,recording_id,diagnosis,predicted_disease_label,predicted_diagnosis
0,101,1b1,URTI,3,COPD
1,101,1b1,URTI,3,COPD
2,101,1b1,URTI,3,COPD
3,101,1b1,URTI,3,COPD
4,101,1b1,URTI,3,COPD
5,101,1b1,URTI,3,COPD
6,101,1b1,URTI,3,COPD
7,101,1b1,URTI,3,COPD
8,101,1b1,URTI,3,COPD
9,101,1b1,URTI,3,COPD


In [25]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import torch
import joblib

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

MODELS_DIR = PROJECT_ROOT / "models"
DL_DIR = MODELS_DIR / "deep_learning"
TRADITIONAL_DIR = MODELS_DIR / "traditional_ml"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

cnn_lstm_predictions_path = (
    DL_DIR / "cnn_lstm" / "cnn_lstm_test_predictions.csv"
)

cnn_lstm_df = pd.read_csv(cnn_lstm_predictions_path)

print("Loaded:", cnn_lstm_predictions_path)
print("Shape:", cnn_lstm_df.shape)
print("Columns:", cnn_lstm_df.columns.tolist())


Loaded: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv
Shape: (1561, 10)
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'predicted_disease_label', 'predicted_diagnosis']


In [26]:

class_names = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

print("Diagnosis vs disease_label:")
display(
    cnn_lstm_df.groupby(["diagnosis", "disease_label"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["diagnosis", "disease_label"])
)

print("\nPredicted label vs predicted diagnosis:")
display(
    cnn_lstm_df.groupby(
        ["predicted_disease_label", "predicted_diagnosis"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
    .sort_values(["predicted_disease_label", "predicted_diagnosis"])
)

print("\nUnique true numeric labels:")
print(sorted(cnn_lstm_df["disease_label"].unique()))

print("\nUnique predicted numeric labels:")
print(sorted(cnn_lstm_df["predicted_disease_label"].unique()))

print("\nExpected mapping:")
display(
    pd.DataFrame({
        "disease_label": range(len(class_names)),
        "expected_diagnosis": class_names,
    })
)


Diagnosis vs disease_label:


,diagnosis,disease_label,rows
0,Bronchiectasis,1,20
1,Bronchiolitis,2,19
2,COPD,3,1337
3,Healthy,4,84
4,Pneumonia,6,52
5,URTI,7,49



Predicted label vs predicted diagnosis:


,predicted_disease_label,predicted_diagnosis,rows
0,0,Asthma,35
1,3,COPD,1411
2,6,Pneumonia,115



Unique true numeric labels:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7)]

Unique predicted numeric labels:
[np.int64(0), np.int64(3), np.int64(6)]

Expected mapping:


,disease_label,expected_diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI


In [27]:

print("Total prediction rows:", len(cnn_lstm_df))

cycle_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
]

print(
    "Duplicate patient-recording-location rows:",
    cnn_lstm_df.duplicated(cycle_keys).sum()
)

print(
    "Unique patient-recording-location combinations:",
    cnn_lstm_df[cycle_keys].drop_duplicates().shape[0]
)

print("\nUnique patients:", cnn_lstm_df["patient_id"].nunique())
print("\nTrue class distribution:")
display(
    cnn_lstm_df["diagnosis"]
    .value_counts()
    .rename_axis("diagnosis")
    .reset_index(name="cycles")
)

print("\nPredicted class distribution:")
display(
    cnn_lstm_df["predicted_diagnosis"]
    .value_counts()
    .rename_axis("predicted_diagnosis")
    .reset_index(name="cycles")
)


Total prediction rows: 1561
Duplicate patient-recording-location rows: 1362
Unique patient-recording-location combinations: 199

Unique patients: 25

True class distribution:


,diagnosis,cycles
0,COPD,1337
1,Healthy,84
2,Pneumonia,52
3,URTI,49
4,Bronchiectasis,20
5,Bronchiolitis,19



Predicted class distribution:


,predicted_diagnosis,cycles
0,COPD,1411
1,Pneumonia,115
2,Asthma,35


In [28]:

lightweight_predictions_path = (
    DL_DIR / "lightweight_mel_cnn" / "cnn_test_predictions.csv"
)

lightweight_df = pd.read_csv(lightweight_predictions_path)

print("CNN-LSTM rows:", len(cnn_lstm_df))
print("Lightweight CNN rows:", len(lightweight_df))

comparison_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
]

print(
    "\nCNN-LSTM duplicate keys:",
    cnn_lstm_df.duplicated(comparison_keys).sum()
)

print(
    "Lightweight CNN duplicate keys:",
    lightweight_df.duplicated(comparison_keys).sum()
)

if "audio_path" in lightweight_df.columns:
    print(
        "\nLightweight CNN unique audio paths:",
        lightweight_df["audio_path"].nunique()
    )

    print(
        "Lightweight CNN duplicate audio paths:",
        lightweight_df["audio_path"].duplicated().sum()
    )

    print("\nExample repeated CNN-LSTM rows:")
    repeated_keys = cnn_lstm_df.loc[
        cnn_lstm_df.duplicated(comparison_keys, keep=False),
        comparison_keys
    ].head(10)

    display(
        cnn_lstm_df.merge(
            repeated_keys.drop_duplicates(),
            on=comparison_keys,
            how="inner"
        ).head(10)
    )


CNN-LSTM rows: 1561
Lightweight CNN rows: 1561

CNN-LSTM duplicate keys: 1362
Lightweight CNN duplicate keys: 1362

Lightweight CNN unique audio paths: 1561
Lightweight CNN duplicate audio paths: 0

Example repeated CNN-LSTM rows:


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split,predicted_disease_label,predicted_diagnosis
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
3,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
4,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
5,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
6,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
7,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
8,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
9,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD


In [29]:

alignment_cols = [
    "patient_id",
    "recording_id",
    "chest_location",
    "diagnosis",
    "disease_label",
]

print("Same number of rows:", len(cnn_lstm_df) == len(lightweight_df))

print(
    "True-label sequences match:",
    cnn_lstm_df["diagnosis"].reset_index(drop=True).equals(
        lightweight_df["diagnosis"].reset_index(drop=True)
    )
)

print(
    "Numeric-label sequences match:",
    cnn_lstm_df["disease_label"].reset_index(drop=True).equals(
        lightweight_df["true_label"].reset_index(drop=True)
    )
)

print(
    "Patient/recording/location sequences match:",
    cnn_lstm_df[alignment_cols[:3]].reset_index(drop=True).equals(
        lightweight_df[alignment_cols[:3]].reset_index(drop=True)
    )
)

print("\nFirst 10 aligned rows:")
display(pd.DataFrame({
    "cnn_lstm_diagnosis": cnn_lstm_df["diagnosis"].head(10),
    "lightweight_diagnosis": lightweight_df["diagnosis"].head(10),
    "cnn_lstm_prediction": cnn_lstm_df["predicted_diagnosis"].head(10),
    "lightweight_prediction": lightweight_df["predicted_disease"].head(10),
}))


Same number of rows: True
True-label sequences match: True
Numeric-label sequences match: True
Patient/recording/location sequences match: True

First 10 aligned rows:


,cnn_lstm_diagnosis,lightweight_diagnosis,cnn_lstm_prediction,lightweight_prediction
0,URTI,URTI,COPD,LRTI
1,URTI,URTI,COPD,Bronchiolitis
2,URTI,URTI,COPD,Bronchiolitis
3,URTI,URTI,COPD,Bronchiolitis
4,URTI,URTI,COPD,Bronchiolitis
5,URTI,URTI,COPD,Healthy
6,URTI,URTI,COPD,Healthy
7,URTI,URTI,COPD,Bronchiolitis
8,URTI,URTI,COPD,Bronchiolitis
9,URTI,URTI,COPD,Bronchiolitis


In [30]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

y_true = lightweight_df["true_label"].astype(int).to_numpy()

y_lightweight = lightweight_df["predicted_label"].astype(int).to_numpy()

y_cnn_lstm = cnn_lstm_df[
    "predicted_disease_label"
].astype(int).to_numpy()

comparison = pd.DataFrame([
    {
        "Model": "Lightweight Mel CNN",
        "Accuracy": accuracy_score(y_true, y_lightweight),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_lightweight
        ),
        "Macro F1": f1_score(
            y_true, y_lightweight, average="macro", zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true, y_lightweight, average="weighted", zero_division=0
        ),
    },
    {
        "Model": "CNN-LSTM",
        "Accuracy": accuracy_score(y_true, y_cnn_lstm),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_cnn_lstm
        ),
        "Macro F1": f1_score(
            y_true, y_cnn_lstm, average="macro", zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true, y_cnn_lstm, average="weighted", zero_division=0
        ),
    },
])

display(comparison.round(4))

print("\nCNN-LSTM classification report:")
print(classification_report(
    y_true,
    y_cnn_lstm,
    labels=list(range(8)),
    target_names=class_names,
    zero_division=0,
))

print("\nCNN-LSTM confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_true, y_cnn_lstm, labels=list(range(8))),
    index=class_names,
    columns=class_names,
))


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
0,Lightweight Mel CNN,0.7168,0.3102,0.2129,0.7687
1,CNN-LSTM,0.8136,0.2261,0.1674,0.7867



CNN-LSTM classification report:
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00        20
 Bronchiolitis       0.00      0.00      0.00        19
          COPD       0.88      0.93      0.91      1337
       Healthy       0.00      0.00      0.00        84
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.19      0.42      0.26        52
          URTI       0.00      0.00      0.00        49

      accuracy                           0.81      1561
     macro avg       0.13      0.17      0.15      1561
  weighted avg       0.76      0.81      0.79      1561


CNN-LSTM confusion matrix:


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,0,0,0,0,0
Bronchiectasis,0,0,0,20,0,0,0,0
Bronchiolitis,0,0,0,17,0,0,2,0
COPD,24,0,0,1248,0,0,65,0
Healthy,7,0,0,60,0,0,17,0
LRTI,0,0,0,0,0,0,0,0
Pneumonia,4,0,0,26,0,0,22,0
URTI,0,0,0,40,0,0,9,0


In [31]:

cnn_lstm_checkpoint_path = (
    DL_DIR / "cnn_lstm" / "best_cnn_lstm.pt"
)

checkpoint = torch.load(
    cnn_lstm_checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    print("\nState-dict tensor count:", len(checkpoint))

    print("\nTensor names and shapes:")
    for name, value in checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(f"{name}: {tuple(value.shape)}")


Checkpoint type: OrderedDict
Top-level keys: ['cnn.0.weight', 'cnn.0.bias', 'cnn.1.weight', 'cnn.1.bias', 'cnn.1.running_mean', 'cnn.1.running_var', 'cnn.1.num_batches_tracked', 'cnn.4.weight', 'cnn.4.bias', 'cnn.5.weight', 'cnn.5.bias', 'cnn.5.running_mean', 'cnn.5.running_var', 'cnn.5.num_batches_tracked', 'cnn.8.weight', 'cnn.8.bias', 'cnn.9.weight', 'cnn.9.bias', 'cnn.9.running_mean', 'cnn.9.running_var', 'cnn.9.num_batches_tracked', 'lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l1', 'lstm.weight_hh_l1', 'lstm.bias_ih_l1', 'lstm.bias_hh_l1', 'classifier.weight', 'classifier.bias']

State-dict tensor count: 31

Tensor names and shapes:
cnn.0.weight: (16, 1, 3, 3)
cnn.0.bias: (16,)
cnn.1.weight: (16,)
cnn.1.bias: (16,)
cnn.1.running_mean: (16,)
cnn.1.running_var: (16,)
cnn.1.num_batches_tracked: ()
cnn.4.weight: (32, 16, 3, 3)
cnn.4.bias: (32,)
cnn.5.weight: (32,)
cnn.5.bias: (32,)
cnn.5.running_mean: (32,)
cnn.5.running_var: (32,)
cn

In [32]:

import torch.nn as nn

class CNNLSTMValidation(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.cnn(x)

        # [batch, channels, mel, time] -> [batch, time, channels]
        x = x.mean(dim=2).transpose(1, 2)

        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        x = self.dropout(x)

        return self.classifier(x)


cnn_lstm_model = CNNLSTMValidation(num_classes=8)

load_result = cnn_lstm_model.load_state_dict(
    checkpoint,
    strict=True,
)

cnn_lstm_model.eval()

print("Checkpoint loaded successfully.")
print("Strict loading result:", load_result)
print(
    "Total parameters:",
    sum(p.numel() for p in cnn_lstm_model.parameters())
)


Checkpoint loaded successfully.
Strict loading result: <All keys matched successfully>
Total parameters: 90600


In [ ]:

import nbformat

training_notebook_path = (
    PROJECT_ROOT / "notebooks" / "13_cnn_lstm_optional.ipynb"
)

training_nb = nbformat.read(
    training_notebook_path,
    as_version=4,
)

search_terms = [
    "melspectrogram",
    "power_to_db",
    "n_fft",
    "hop_length",
    "n_mels",
    "fmin",
    "fmax",
    "normalization",
    "normalize",
    "logmel_train_normalization",
    "audio_path",
    "resample",
    "sample_rate",
    "20000",
]

print("Notebook:", training_notebook_path)
print("Total cells:", len(training_nb.cells))

for i, cell in enumerate(training_nb.cells):
    if cell.cell_type != "code":
        continue

    source = cell.source
    matched_terms = [
        term for term in search_terms
        if term.lower() in source.lower()
    ]

    if matched_terms:
        print(f"\n{'=' * 70}")
        print(f"CELL {i} | Matches: {', '.join(matched_terms)}")
        print(source[:6000])


In [33]:

import json
from pathlib import Path

training_notebook_path = (
    PROJECT_ROOT / "notebooks" / "13_cnn_lstm_optional.ipynb"
)

print("Notebook exists:", training_notebook_path.exists())

if training_notebook_path.exists():
    with open(training_notebook_path, "r", encoding="utf-8") as f:
        training_notebook = json.load(f)

    keywords = [
        "librosa",
        "melspectrogram",
        "power_to_db",
        "n_fft",
        "hop_length",
        "n_mels",
        "StandardScaler",
        "normalization",
        "mean(",
        "std(",
        "audio_path",
        "CNNLSTM",
        "DataLoader",
        "train_dataset",
        "test_dataset",
        "torch.save",
        "predicted_disease_label",
    ]

    for cell_number, cell in enumerate(
        training_notebook.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            print(f"\n{'=' * 70}")
            print(f"Cell {cell_number} | Matches: {', '.join(matched)}")
            print(source[:7000])


Notebook exists: True

Cell 2 | Matches: DataLoader
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch version:", torch.__version__)

Cell 10 | Matches: n_fft, hop_length, n_mels
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
FMIN = 0
FMAX = 2000

print("Target sample rate:", TARGET_SR)
print("Target duration:", TARGET_DURATION)
print("Target samples:", TARGET_SAMPLES)
print("Log-Mel configuration:")
print(f"  N_FFT: {N_FFT}")
print(f"  Hop length: {HOP_LENGTH}")
print(f"  Mel bands: {N_MELS}")
print(f"  Frequency ran

In [34]:

cnn_lstm_dir = DL_DIR / "cnn_lstm"

normalization_path = (
    cnn_lstm_dir / "logmel_train_normalization.csv"
)

normalization_df = pd.read_csv(normalization_path)

display(normalization_df)

logmel_mean = float(
    normalization_df.loc[
        normalization_df["statistic"] == "mean", "value"
    ].iloc[0]
)

logmel_std = float(
    normalization_df.loc[
        normalization_df["statistic"] == "std", "value"
    ].iloc[0]
)

print("Log-Mel mean:", logmel_mean)
print("Log-Mel std:", logmel_std)

assert np.isfinite(logmel_mean)
assert np.isfinite(logmel_std)
assert logmel_std > 0

print("\nSaved prediction columns:")
print(cnn_lstm_df.columns.tolist())

print("\nFirst audio path from aligned test data:")
print(lightweight_df["audio_path"].iloc[0])


,statistic,value
0,mean,-66.740814
1,std,17.337090


Log-Mel mean: -66.74081420898438
Log-Mel std: 17.33708953857422

Saved prediction columns:
['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'predicted_disease_label', 'predicted_diagnosis']

First audio path from aligned test data:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_001.wav


In [36]:

print("Audio path:", audio_path)
print("Sampling rate:", sr)
print("Input tensor shape:", tuple(sample_tensor.shape))
print("Input finite:", torch.isfinite(sample_tensor).all().item())
print("True diagnosis:", sample_row["diagnosis"])
print("Saved CNN-LSTM prediction:", cnn_lstm_df.iloc[0]["predicted_diagnosis"])
print("Saved Lightweight CNN prediction:", sample_row["predicted_disease"])
print("New CNN-LSTM prediction:", class_names[predicted_label])
print("Maximum probability:", float(probabilities.max().item()))


Audio path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_001.wav
Sampling rate: 4000
Input tensor shape: (1, 1, 64, 157)
Input finite: True
True diagnosis: URTI
Saved CNN-LSTM prediction: COPD
Saved Lightweight CNN prediction: LRTI
New CNN-LSTM prediction: COPD
Maximum probability: 0.6078786849975586


In [37]:

from torch.utils.data import Dataset, DataLoader

class CNNLSTMInferenceDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        signal, sr = sf.read(row["audio_path"])

        if signal.ndim > 1:
            signal = np.mean(signal, axis=1)

        if sr != TARGET_SR:
            raise ValueError(
                f"Expected {TARGET_SR} Hz, got {sr} Hz"
            )

        signal = signal.astype(np.float32)

        if len(signal) < TARGET_SAMPLES:
            signal = np.pad(
                signal,
                (0, TARGET_SAMPLES - len(signal)),
                mode="constant",
            )
        else:
            signal = signal[:TARGET_SAMPLES]

        mel_power = librosa.feature.melspectrogram(
            y=signal,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0,
        )

        mel_db = librosa.power_to_db(mel_power, ref=np.max)
        mel_normalized = (mel_db - logmel_mean) / logmel_std

        x = torch.tensor(
            mel_normalized,
            dtype=torch.float32,
        ).unsqueeze(0)

        return x


inference_dataset = CNNLSTMInferenceDataset(lightweight_df)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

all_predictions = []

cnn_lstm_model.eval()

with torch.no_grad():
    for batch in inference_loader:
        logits = cnn_lstm_model(batch)
        all_predictions.extend(
            logits.argmax(dim=1).cpu().numpy().tolist()
        )

all_predictions = np.asarray(all_predictions, dtype=int)

saved_predictions = (
    cnn_lstm_df["predicted_disease_label"].astype(int).to_numpy()
)

print("Reproduced predictions:", len(all_predictions))
print("Saved predictions:", len(saved_predictions))
print(
    "Exact prediction agreement:",
    np.mean(all_predictions == saved_predictions)
)
print(
    "Number of mismatches:",
    int(np.sum(all_predictions != saved_predictions))
)


Reproduced predictions: 1561
Saved predictions: 1561
Exact prediction agreement: 1.0
Number of mismatches: 0


In [38]:

import json

lightweight_notebook_path = (
    PROJECT_ROOT / "notebooks" / "11_cnn_baseline.ipynb"
)

print("Notebook exists:", lightweight_notebook_path.exists())

if lightweight_notebook_path.exists():
    with open(lightweight_notebook_path, "r", encoding="utf-8") as f:
        lightweight_notebook = json.load(f)

    keywords = [
        "class LightweightMelCNN",
        "def forward",
        "melspectrogram",
        "power_to_db",
        "normalization",
        "logmel_mean",
        "logmel_std",
        "CNNBaselineDataset",
        "Dataset",
        "DataLoader",
        "torch.save",
        "test_predictions",
        "model.load_state_dict",
    ]

    for cell_number, cell in enumerate(
        lightweight_notebook.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            print(f"\n{'=' * 70}")
            print(f"Cell {cell_number} | Matches: {', '.join(matched)}")
            print(source[:6000])


Notebook exists: True

Cell 2 | Matches: Dataset, DataLoader
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("PyTorch version:", torch.__version__)

Cell 5 | Matches: Dataset
patient_diagnosis = (
    dl_manifest
    .groupby("patient_id")["diagnosis"]
    .nunique()
)

print(
    "Patients with exactly one diagnosis:",
    (patient_diagnosis == 1).sum()
)

print(
    "Patients with multiple diagnoses:",
    (patient_diagnosis > 1).sum()
)

assert (patient_diagnosis == 1).all()

patient_df = (
    dl_manifest[
        ["patient_id", "diagnosis"]
    ]
    .drop_duplicates()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

p

In [39]:

import torch.nn as nn

class LightweightMelCNNValidation(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


lightweight_checkpoint_path = (
    DL_DIR / "lightweight_mel_cnn" / "best_lightweight_mel_cnn.pt"
)

lightweight_checkpoint = torch.load(
    lightweight_checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

print("Checkpoint keys:", list(lightweight_checkpoint.keys()))

lightweight_model = LightweightMelCNNValidation(num_classes=8)

load_result = lightweight_model.load_state_dict(
    lightweight_checkpoint["model_state_dict"],
    strict=True,
)

lightweight_model.eval()

print("Strict loading:", load_result)
print(
    "Parameter count:",
    sum(p.numel() for p in lightweight_model.parameters())
)
print("Saved checkpoint epoch:", lightweight_checkpoint["epoch"])
print("Saved validation loss:", lightweight_checkpoint["val_loss"])


Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_accuracy', 'class_mapping', 'config']
Strict loading: <All keys matched successfully>
Parameter count: 24040
Saved checkpoint epoch: 32
Saved validation loss: 1.2070167528983018


In [40]:

lightweight_normalization_path = (
    DL_DIR / "lightweight_mel_cnn" / "logmel_train_normalization.csv"
)

lightweight_norm_df = pd.read_csv(lightweight_normalization_path)

lightweight_mean = float(
    lightweight_norm_df.loc[
        lightweight_norm_df["statistic"] == "mean", "value"
    ].iloc[0]
)

lightweight_std = float(
    lightweight_norm_df.loc[
        lightweight_norm_df["statistic"] == "std", "value"
    ].iloc[0]
)

print("Training mean:", lightweight_mean)
print("Training std:", lightweight_std)

# Reuse the test cycles in the exact saved order.
inference_dataset = CNNLSTMInferenceDataset(lightweight_df)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

# CNN-LSTMInferenceDataset uses the CNN-LSTM normalization globals.
# Update them to the Lightweight CNN's saved training statistics.
# Restore the originals afterward.
original_mean = logmel_mean
original_std = logmel_std

try:
    logmel_mean = lightweight_mean
    logmel_std = lightweight_std

    all_lightweight_predictions = []

    lightweight_model.eval()

    with torch.no_grad():
        for batch in inference_loader:
            logits = lightweight_model(batch)
            all_lightweight_predictions.extend(
                logits.argmax(dim=1).cpu().numpy().tolist()
            )
finally:
    logmel_mean = original_mean
    logmel_std = original_std

all_lightweight_predictions = np.asarray(
    all_lightweight_predictions, dtype=int
)

saved_lightweight_predictions = (
    lightweight_df["predicted_label"].astype(int).to_numpy()
)

print("\nRegenerated predictions:", len(all_lightweight_predictions))
print("Saved predictions:", len(saved_lightweight_predictions))
print(
    "Exact prediction agreement:",
    np.mean(
        all_lightweight_predictions == saved_lightweight_predictions
    )
)
print(
    "Number of mismatches:",
    int(np.sum(
        all_lightweight_predictions != saved_lightweight_predictions
    ))
)


Training mean: -66.74081420898438
Training std: 17.33708953857422

Regenerated predictions: 1561
Saved predictions: 1561
Exact prediction agreement: 1.0
Number of mismatches: 0


In [41]:

multifeature_notebook_path = (
    PROJECT_ROOT / "notebooks" / "12_multifeature_cnn.ipynb"
)

print("Notebook exists:", multifeature_notebook_path.exists())

if multifeature_notebook_path.exists():
    with open(multifeature_notebook_path, "r", encoding="utf-8") as f:
        multifeature_notebook = json.load(f)

    keywords = [
        "class FeatureBranch",
        "class MultiFeatureCNN",
        "def forward",
        "melspectrogram",
        "mfcc",
        "chroma",
        "normalization",
        "train_mean",
        "train_std",
        "Dataset",
        "torch.save",
        "test_predictions",
        "model.load_state_dict",
    ]

    for cell_number, cell in enumerate(
        multifeature_notebook.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            print(f"\n{'=' * 70}")
            print(f"Cell {cell_number} | Matches: {', '.join(matched)}")
            print(source[:6000])


Notebook exists: True

Cell 2 | Matches: Dataset
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("Librosa version:", librosa.__version__)

Cell 13 | Matches: melspectrogram
def extract_logmel(signal):
    mel_power = librosa.feature.melspectrogram(
        y=signal,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel_power

In [42]:

class FeatureBranchValidation(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, output_dim),
            nn.ReLU(),
            nn.Dropout(0.30),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class MultiFeatureCNNValidation(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.logmel_branch = FeatureBranchValidation(64)
        self.mfcc_branch = FeatureBranchValidation(64)
        self.chroma_branch = FeatureBranchValidation(64)

        self.classifier = nn.Sequential(
            nn.Linear(64 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        logmel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        fused = torch.cat(
            [logmel_features, mfcc_features, chroma_features],
            dim=1,
        )

        return self.classifier(fused)


multifeature_checkpoint_path = (
    DL_DIR / "multifeature_cnn" / "best_multifeature_cnn.pt"
)

multifeature_checkpoint = torch.load(
    multifeature_checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

multifeature_model = MultiFeatureCNNValidation(num_classes=8)

load_result = multifeature_model.load_state_dict(
    multifeature_checkpoint,
    strict=True,
)

multifeature_model.eval()

print("Checkpoint loaded:", multifeature_checkpoint_path)
print("Strict loading:", load_result)
print(
    "Parameter count:",
    sum(p.numel() for p in multifeature_model.parameters()),
)


Checkpoint loaded: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/multifeature_cnn/best_multifeature_cnn.pt
Strict loading: <All keys matched successfully>
Parameter count: 108776


In [43]:

multifeature_normalization_path = (
    DL_DIR / "multifeature_cnn" / "multifeature_train_normalization.csv"
)

multifeature_norm_df = pd.read_csv(
    multifeature_normalization_path
)

display(multifeature_norm_df)

required_features = {"logmel", "mfcc", "chroma"}

assert required_features.issubset(
    set(multifeature_norm_df["feature"])
), "Missing feature normalization statistics."

assert np.isfinite(multifeature_norm_df["mean"]).all()
assert np.isfinite(multifeature_norm_df["std"]).all()
assert (multifeature_norm_df["std"] > 0).all()

feature_normalization = {
    row["feature"]: {
        "mean": float(row["mean"]),
        "std": float(row["std"]),
    }
    for _, row in multifeature_norm_df.iterrows()
}

print("\nVerified normalization values:")
for feature, stats in feature_normalization.items():
    print(
        f"{feature}: mean={stats['mean']:.8f}, "
        f"std={stats['std']:.8f}"
    )


,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059



Verified normalization values:
logmel: mean=-66.23048066, std=17.66188608
mfcc: mean=-22.50903445, std=137.27441616
chroma: mean=0.38565572, std=0.38905858


In [44]:

multifeature_predictions_path = (
    DL_DIR / "multifeature_cnn" / "multifeature_cnn_test_predictions.csv"
)

multifeature_predictions_df = pd.read_csv(
    multifeature_predictions_path
)

print("Saved prediction shape:", multifeature_predictions_df.shape)
print("Test dataframe shape:", lightweight_df.shape)

print("\nSaved prediction columns:")
print(multifeature_predictions_df.columns.tolist())

print("\nSaved true-label distribution:")
print(multifeature_predictions_df["true_label"].value_counts().sort_index())

print("\nReference true-label distribution:")
print(lightweight_df["true_label"].value_counts().sort_index())

assert len(multifeature_predictions_df) == len(lightweight_df), (
    "Row counts differ; investigate the test split before inference."
)

assert (
    multifeature_predictions_df["true_label"].to_numpy()
    == lightweight_df["true_label"].to_numpy()
).all(), "True-label order differs between the saved predictions and reference dataframe."

assert lightweight_df["audio_path"].notna().all()
assert lightweight_df["audio_path"].map(lambda p: Path(p).is_file()).all()

print("\nPASS: Row counts, true-label order, and audio paths are verified.")


Saved prediction shape: (1561, 10)
Test dataframe shape: (1561, 12)

Saved prediction columns:
['true_label', 'predicted_label', 'prob_Asthma', 'prob_Bronchiectasis', 'prob_Bronchiolitis', 'prob_COPD', 'prob_Healthy', 'prob_LRTI', 'prob_Pneumonia', 'prob_URTI']

Saved true-label distribution:
true_label
1      20
2      19
3    1337
4      84
6      52
7      49
Name: count, dtype: int64

Reference true-label distribution:
true_label
1      20
2      19
3    1337
4      84
6      52
7      49
Name: count, dtype: int64

PASS: Row counts, true-label order, and audio paths are verified.


In [45]:

from torch.utils.data import Dataset, DataLoader
import soundfile as sf
import librosa

class MultiFeatureInferenceDataset(Dataset):
    def __init__(self, dataframe, normalization):
        self.df = dataframe.reset_index(drop=True)
        self.normalization = normalization

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        signal, sr = sf.read(row["audio_path"])

        if signal.ndim > 1:
            signal = np.mean(signal, axis=1)

        if sr != TARGET_SR:
            raise ValueError(
                f"Unexpected sample rate {sr} for {row['audio_path']}"
            )

        signal = signal.astype(np.float32)

        if len(signal) < TARGET_SAMPLES:
            signal = np.pad(
                signal,
                (0, TARGET_SAMPLES - len(signal))
            )
        else:
            signal = signal[:TARGET_SAMPLES]

        # 1. Log-Mel spectrogram
        mel_power = librosa.feature.melspectrogram(
            y=signal,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0
        )

        logmel = librosa.power_to_db(
            mel_power,
            ref=np.max
        ).astype(np.float32)

        # 2. MFCC
        mfcc = librosa.feature.mfcc(
            y=signal,
            sr=TARGET_SR,
            n_mfcc=13,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            fmin=FMIN,
            fmax=FMAX
        ).astype(np.float32)

        # 3. Chroma
        chroma = librosa.feature.chroma_stft(
            y=signal,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_chroma=12,
            tuning=0.0
        ).astype(np.float32)

        # Apply saved training-set normalization
        logmel = (
            logmel - self.normalization["logmel"]["mean"]
        ) / self.normalization["logmel"]["std"]

        mfcc = (
            mfcc - self.normalization["mfcc"]["mean"]
        ) / self.normalization["mfcc"]["std"]

        chroma = (
            chroma - self.normalization["chroma"]["mean"]
        ) / self.normalization["chroma"]["std"]

        # Add channel dimension: [1, frequency, time]
        logmel_tensor = torch.from_numpy(
            logmel[np.newaxis, :, :].astype(np.float32)
        )
        mfcc_tensor = torch.from_numpy(
            mfcc[np.newaxis, :, :].astype(np.float32)
        )
        chroma_tensor = torch.from_numpy(
            chroma[np.newaxis, :, :].astype(np.float32)
        )

        return logmel_tensor, mfcc_tensor, chroma_tensor


multifeature_test_dataset = MultiFeatureInferenceDataset(
    lightweight_df,
    feature_normalization
)

print("Test cycles:", len(multifeature_test_dataset))

sample_logmel, sample_mfcc, sample_chroma = (
    multifeature_test_dataset[0]
)

print("Log-Mel shape:", tuple(sample_logmel.shape))
print("MFCC shape:", tuple(sample_mfcc.shape))
print("Chroma shape:", tuple(sample_chroma.shape))

assert sample_logmel.shape == (1, 64, 157)
assert sample_mfcc.shape == (1, 13, 157)
assert sample_chroma.shape == (1, 12, 157)

assert torch.isfinite(sample_logmel).all()
assert torch.isfinite(sample_mfcc).all()
assert torch.isfinite(sample_chroma).all()

print("\nPASS: Feature shapes and numerical values are valid.")


Test cycles: 1561
Log-Mel shape: (1, 64, 157)
MFCC shape: (1, 13, 157)
Chroma shape: (1, 12, 157)

PASS: Feature shapes and numerical values are valid.


In [46]:

multifeature_test_loader = DataLoader(
    multifeature_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

multifeature_model.eval()
multifeature_model.to(DEVICE)

multifeature_recomputed_predictions = []

with torch.no_grad():
    for logmel_batch, mfcc_batch, chroma_batch in multifeature_test_loader:
        logmel_batch = logmel_batch.to(DEVICE)
        mfcc_batch = mfcc_batch.to(DEVICE)
        chroma_batch = chroma_batch.to(DEVICE)

        logits = multifeature_model(
            logmel_batch,
            mfcc_batch,
            chroma_batch
        )

        predicted_labels = logits.argmax(dim=1)
        multifeature_recomputed_predictions.extend(
            predicted_labels.cpu().numpy().tolist()
        )

multifeature_recomputed_predictions = np.asarray(
    multifeature_recomputed_predictions
)

multifeature_saved_predictions = (
    multifeature_predictions_df["predicted_label"]
    .to_numpy(dtype=int)
)

assert len(multifeature_recomputed_predictions) == len(
    multifeature_saved_predictions
)

multifeature_matches = (
    multifeature_recomputed_predictions
    == multifeature_saved_predictions
)

multifeature_agreement = multifeature_matches.mean()
multifeature_mismatches = int((~multifeature_matches).sum())

print("Test cycles evaluated:", len(multifeature_recomputed_predictions))
print(f"Prediction agreement: {multifeature_agreement:.6f}")
print("Matching predictions:", int(multifeature_matches.sum()))
print("Mismatched predictions:", multifeature_mismatches)

if multifeature_mismatches == 0:
    print("\nPASS: Recomputed predictions exactly match saved predictions.")
else:
    print("\nINVESTIGATE: Recomputed predictions differ from saved predictions.")
    mismatch_indices = np.where(~multifeature_matches)[0][:10]
    print("First mismatching row indices:", mismatch_indices.tolist())
    print(
        "Recomputed labels:",
        multifeature_recomputed_predictions[mismatch_indices].tolist()
    )
    print(
        "Saved labels:",
        multifeature_saved_predictions[mismatch_indices].tolist()
    )


Test cycles evaluated: 1561
Prediction agreement: 1.000000
Matching predictions: 1561
Mismatched predictions: 0

PASS: Recomputed predictions exactly match saved predictions.


In [47]:

regularized_dir = DL_DIR / "regularized_multifeature_cnn"

regularized_checkpoint_path = (
    regularized_dir / "best_regularized_multifeature_cnn.pt"
)
regularized_config_path = regularized_dir / "training_config.json"
regularized_predictions_path = regularized_dir / "test_predictions.csv"

print("Checkpoint exists:", regularized_checkpoint_path.is_file())
print("Training config exists:", regularized_config_path.is_file())
print("Predictions exist:", regularized_predictions_path.is_file())

print("\nFiles in the regularized model directory:")
for path in sorted(regularized_dir.iterdir()):
    print(path.name)

if regularized_config_path.is_file():
    with open(regularized_config_path, "r") as f:
        regularized_config = json.load(f)

    print("\nTraining configuration:")
    print(json.dumps(regularized_config, indent=2))

if regularized_predictions_path.is_file():
    regularized_predictions_df = pd.read_csv(
        regularized_predictions_path
    )

    print("\nSaved predictions shape:", regularized_predictions_df.shape)
    print("Prediction columns:")
    print(regularized_predictions_df.columns.tolist())

    print("\nFirst five rows:")
    display(regularized_predictions_df.head())

    assert len(regularized_predictions_df) == len(lightweight_df), (
        "12B prediction row count differs from the reference test dataframe."
    )

    assert (
        regularized_predictions_df["true_label"].to_numpy()
        == lightweight_df["true_label"].to_numpy()
    ).all(), "12B true-label ordering differs from the reference test dataframe."

    print("\nPASS: 12B prediction rows and true-label ordering match.")


Checkpoint exists: True
Training config exists: True
Predictions exist: True

Files in the regularized model directory:
best_regularized_multifeature_cnn.pt
test_predictions.csv
training_config.json
training_history.csv

Training configuration:
{
  "experiment": "12B",
  "model": "Regularized Multi-Feature CNN",
  "features": [
    "Log-Mel",
    "MFCC",
    "Chroma"
  ],
  "patient_disjoint": true,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "dropout_branch": 0.5,
  "dropout_classifier": 0.5,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "scheduler": "ReduceLROnPlateau",
  "scheduler_factor": 0.5,
  "scheduler_patience": 3,
  "min_learning_rate": 1e-06,
  "loss": "weighted_cross_entropy",
  "augmentation": [
    "Gaussian noise",
    "Random time shift"
  ]
}

Saved predictions shape: (1561, 10)
Prediction columns:
['true_label', 'predicted_label', 'prob_Asthma', 'prob_Bronchiectasis', 'prob_Bronchiolitis', 'prob_COPD', 'prob_Healthy', 'prob_LRTI', 'prob_Pneumonia',

,true_label,predicted_label,prob_Asthma,prob_Bronchiectasis,prob_Bronchiolitis,prob_COPD,prob_Healthy,prob_LRTI,prob_Pneumonia,prob_URTI
0,7,3,0.006300,0.136707,0.065136,0.602875,0.039925,0.019768,0.097903,0.031387
1,7,3,0.012703,0.130662,0.073895,0.542733,0.053770,0.029238,0.115558,0.041441
2,7,3,0.010876,0.132233,0.073360,0.554080,0.050549,0.027199,0.112058,0.039646
3,7,3,0.013296,0.129318,0.074833,0.538601,0.054973,0.030079,0.116749,0.042151
4,7,3,0.012131,0.129467,0.073443,0.548550,0.052822,0.028444,0.114607,0.040535



PASS: 12B prediction rows and true-label ordering match.


In [48]:

# Inspect the saved checkpoint structure without loading it into a model.
regularized_checkpoint = torch.load(
    regularized_checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(regularized_checkpoint).__name__)

if isinstance(regularized_checkpoint, dict):
    print("Top-level keys:", list(regularized_checkpoint.keys()))

    if "model_state_dict" in regularized_checkpoint:
        regularized_state_dict = regularized_checkpoint["model_state_dict"]
    else:
        regularized_state_dict = regularized_checkpoint

    print("\nState-dict tensor count:", len(regularized_state_dict))
    print("\nFirst 15 parameter names and shapes:")

    for i, (name, tensor) in enumerate(regularized_state_dict.items()):
        if i == 15:
            break
        print(name, tuple(tensor.shape))

# Find normalization-related artifacts in the model directories.
print("\nNormalization CSV files:")
for path in sorted(DL_DIR.rglob("*normalization*.csv")):
    print(path.relative_to(DL_DIR))

# Check the original Notebook 12 source for 12B's training and normalization setup.
notebook_12_path = PROJECT_ROOT / "notebooks" / "12_multifeature_cnn.ipynb"

with open(notebook_12_path, "r", encoding="utf-8") as f:
    notebook_12 = json.load(f)

print("\nRelevant Notebook 12 code cells:")

keywords = (
    "RegularizedFeatureBranch",
    "RegularizedMultiFeatureCNN",
    "multifeature_train_normalization",
    "MultiFeatureRespiratoryDataset",
    "train_loader",
    "normalization"
)

for cell_index, cell in enumerate(notebook_12["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print(f"\n--- Cell {cell_index} ---")
        print(source[:5000])


Checkpoint type: OrderedDict
Top-level keys: ['logmel_branch.features.0.weight', 'logmel_branch.features.0.bias', 'logmel_branch.features.1.weight', 'logmel_branch.features.1.bias', 'logmel_branch.features.1.running_mean', 'logmel_branch.features.1.running_var', 'logmel_branch.features.1.num_batches_tracked', 'logmel_branch.features.4.weight', 'logmel_branch.features.4.bias', 'logmel_branch.features.5.weight', 'logmel_branch.features.5.bias', 'logmel_branch.features.5.running_mean', 'logmel_branch.features.5.running_var', 'logmel_branch.features.5.num_batches_tracked', 'logmel_branch.features.8.weight', 'logmel_branch.features.8.bias', 'logmel_branch.features.9.weight', 'logmel_branch.features.9.bias', 'logmel_branch.features.9.running_mean', 'logmel_branch.features.9.running_var', 'logmel_branch.features.9.num_batches_tracked', 'logmel_branch.projection.1.weight', 'logmel_branch.projection.1.bias', 'mfcc_branch.features.0.weight', 'mfcc_branch.features.0.bias', 'mfcc_branch.features.1

In [49]:

class RegularizedFeatureBranchValidation(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, output_dim),
            nn.ReLU(),
            nn.Dropout(0.50)
        )

    def forward(self, x):
        return self.projection(self.features(x))


class RegularizedMultiFeatureCNNValidation(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.logmel_branch = RegularizedFeatureBranchValidation(64)
        self.mfcc_branch = RegularizedFeatureBranchValidation(64)
        self.chroma_branch = RegularizedFeatureBranchValidation(64)

        self.classifier = nn.Sequential(
            nn.Linear(64 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.50),
            nn.Linear(128, num_classes)
        )

    def forward(self, logmel, mfcc, chroma):
        logmel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        fused_features = torch.cat(
            [logmel_features, mfcc_features, chroma_features],
            dim=1
        )

        return self.classifier(fused_features)


regularized_model = RegularizedMultiFeatureCNNValidation(
    num_classes=8
)

regularized_load_result = regularized_model.load_state_dict(
    regularized_checkpoint,
    strict=True
)

regularized_model = regularized_model.to(DEVICE)
regularized_model.eval()

regularized_parameter_count = sum(
    p.numel() for p in regularized_model.parameters()
)

print("Checkpoint:", regularized_checkpoint_path)
print("Strict loading:", regularized_load_result)
print("Parameter count:", regularized_parameter_count)
print("Model is in evaluation mode:", not regularized_model.training)

assert regularized_parameter_count == 108776
print("\nPASS: 12B architecture matches the checkpoint.")


Checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/regularized_multifeature_cnn/best_regularized_multifeature_cnn.pt
Strict loading: <All keys matched successfully>
Parameter count: 108776
Model is in evaluation mode: True

PASS: 12B architecture matches the checkpoint.


In [50]:

regularized_test_predictions = []

regularized_model.eval()

with torch.no_grad():
    for logmel_batch, mfcc_batch, chroma_batch in multifeature_test_loader:
        logmel_batch = logmel_batch.to(DEVICE)
        mfcc_batch = mfcc_batch.to(DEVICE)
        chroma_batch = chroma_batch.to(DEVICE)

        logits = regularized_model(
            logmel_batch,
            mfcc_batch,
            chroma_batch
        )

        predicted_labels = logits.argmax(dim=1)

        regularized_test_predictions.extend(
            predicted_labels.cpu().numpy().tolist()
        )

regularized_test_predictions = np.asarray(
    regularized_test_predictions
)

regularized_saved_labels = (
    regularized_predictions_df["predicted_label"]
    .to_numpy(dtype=int)
)

assert len(regularized_test_predictions) == len(regularized_saved_labels)

regularized_matches = (
    regularized_test_predictions == regularized_saved_labels
)

regularized_agreement = regularized_matches.mean()
regularized_mismatches = int((~regularized_matches).sum())

print("Test cycles evaluated:", len(regularized_test_predictions))
print(f"Prediction agreement: {regularized_agreement:.6f}")
print("Matching predictions:", int(regularized_matches.sum()))
print("Mismatched predictions:", regularized_mismatches)

if regularized_mismatches == 0:
    print("\nPASS: 12B predictions exactly match the saved predictions.")
else:
    print("\nINVESTIGATE: 12B predictions differ from the saved predictions.")

    mismatch_indices = np.where(~regularized_matches)[0][:10]

    print("First mismatching row indices:", mismatch_indices.tolist())
    print(
        "Recomputed labels:",
        regularized_test_predictions[mismatch_indices].tolist()
    )
    print(
        "Saved labels:",
        regularized_saved_labels[mismatch_indices].tolist()
    )


Test cycles evaluated: 1561
Prediction agreement: 1.000000
Matching predictions: 1561
Mismatched predictions: 0

PASS: 12B predictions exactly match the saved predictions.


In [51]:

probability_columns = [
    f"prob_{name}"
    for name in [
        "Asthma",
        "Bronchiectasis",
        "Bronchiolitis",
        "COPD",
        "Healthy",
        "LRTI",
        "Pneumonia",
        "URTI"
    ]
]

def validate_probability_table(df, model_name):
    probabilities = df[probability_columns].to_numpy(dtype=float)

    finite = np.isfinite(probabilities).all()
    within_range = (
        (probabilities >= -1e-6).all()
        and (probabilities <= 1.0 + 1e-6).all()
    )

    row_sums = probabilities.sum(axis=1)
    sums_close_to_one = np.allclose(
        row_sums,
        1.0,
        atol=1e-4,
        rtol=1e-4
    )

    print(f"\n{model_name}")
    print("Rows:", len(df))
    print("All probabilities finite:", finite)
    print("All probabilities within [0, 1]:", within_range)
    print("All row sums approximately 1:", sums_close_to_one)
    print(f"Minimum row sum: {row_sums.min():.8f}")
    print(f"Maximum row sum: {row_sums.max():.8f}")

    assert finite, f"{model_name}: non-finite probabilities found."
    assert within_range, f"{model_name}: probabilities outside [0, 1]."
    assert sums_close_to_one, f"{model_name}: probability sums are invalid."

    print("PASS")

validate_probability_table(
    multifeature_predictions_df,
    "Multi-feature CNN (12A)"
)

validate_probability_table(
    regularized_predictions_df,
    "Regularized Multi-feature CNN (12B)"
)



Multi-feature CNN (12A)
Rows: 1561
All probabilities finite: True
All probabilities within [0, 1]: True
All row sums approximately 1: True
Minimum row sum: 0.99999983
Maximum row sum: 1.00000019
PASS

Regularized Multi-feature CNN (12B)
Rows: 1561
All probabilities finite: True
All probabilities within [0, 1]: True
All row sums approximately 1: True
Minimum row sum: 0.99999987
Maximum row sum: 1.00000018
PASS


In [52]:

def validate_probability_label_consistency(df, model_name):
    probabilities = df[probability_columns].to_numpy(dtype=float)

    probability_argmax = probabilities.argmax(axis=1)
    saved_labels = df["predicted_label"].to_numpy(dtype=int)

    matches = probability_argmax == saved_labels
    mismatch_count = int((~matches).sum())

    print(f"\n{model_name}")
    print("Total rows:", len(df))
    print("Consistent labels:", int(matches.sum()))
    print("Inconsistent labels:", mismatch_count)
    print(f"Consistency: {matches.mean():.6%}")

    if mismatch_count:
        indices = np.where(~matches)[0][:10]
        print("First mismatching row indices:", indices.tolist())
        print("Probability argmax:", probability_argmax[indices].tolist())
        print("Saved labels:", saved_labels[indices].tolist())

    assert mismatch_count == 0, (
        f"{model_name}: predicted labels do not consistently match "
        "the maximum-probability class."
    )

    print("PASS")


validate_probability_label_consistency(
    multifeature_predictions_df,
    "Multi-feature CNN (12A)"
)

validate_probability_label_consistency(
    regularized_predictions_df,
    "Regularized Multi-feature CNN (12B)"
)



Multi-feature CNN (12A)
Total rows: 1561
Consistent labels: 1561
Inconsistent labels: 0
Consistency: 100.000000%
PASS

Regularized Multi-feature CNN (12B)
Total rows: 1561
Consistent labels: 1561
Inconsistent labels: 0
Consistency: 100.000000%
PASS


In [53]:

expected_class_names = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI"
]

expected_probability_columns = [
    f"prob_{name}" for name in expected_class_names
]

assert probability_columns == expected_probability_columns

true_labels = lightweight_df["true_label"].to_numpy(dtype=int)

assert np.isin(true_labels, np.arange(8)).all()

print("Expected class mapping:")
for class_id, class_name in enumerate(expected_class_names):
    print(f"{class_id}: {class_name}")

print("\nTrue test-label distribution:")
print(
    pd.Series(true_labels)
    .map(dict(enumerate(expected_class_names)))
    .value_counts()
    .reindex(expected_class_names, fill_value=0)
)

def show_prediction_distribution(df, model_name):
    predicted_labels = df["predicted_label"].to_numpy(dtype=int)

    assert np.isin(predicted_labels, np.arange(8)).all(), (
        f"{model_name}: found predicted labels outside 0-7."
    )

    distribution = (
        pd.Series(predicted_labels)
        .value_counts()
        .reindex(range(8), fill_value=0)
    )

    report = pd.DataFrame({
        "class_id": range(8),
        "class_name": expected_class_names,
        "predicted_count": distribution.to_numpy(),
        "predicted_percentage": (
            distribution.to_numpy() / len(df) * 100
        ).round(2)
    })

    print(f"\n{model_name} prediction distribution:")
    display(report)

    print("Valid class IDs:", True)
    print("Total predictions:", int(distribution.sum()))

    assert int(distribution.sum()) == len(df)

show_prediction_distribution(
    multifeature_predictions_df,
    "Multi-feature CNN (12A)"
)

show_prediction_distribution(
    regularized_predictions_df,
    "Regularized Multi-feature CNN (12B)"
)


Expected class mapping:
0: Asthma
1: Bronchiectasis
2: Bronchiolitis
3: COPD
4: Healthy
5: LRTI
6: Pneumonia
7: URTI

True test-label distribution:
Asthma               0
Bronchiectasis      20
Bronchiolitis       19
COPD              1337
Healthy             84
LRTI                 0
Pneumonia           52
URTI                49
Name: count, dtype: int64

Multi-feature CNN (12A) prediction distribution:


,class_id,class_name,predicted_count,predicted_percentage
0,0,Asthma,0,0.00
1,1,Bronchiectasis,1,0.06
2,2,Bronchiolitis,0,0.00
3,3,COPD,1559,99.87
4,4,Healthy,0,0.00
5,5,LRTI,0,0.00
6,6,Pneumonia,1,0.06
7,7,URTI,0,0.00


Valid class IDs: True
Total predictions: 1561

Regularized Multi-feature CNN (12B) prediction distribution:


,class_id,class_name,predicted_count,predicted_percentage
0,0,Asthma,0,0.0
1,1,Bronchiectasis,0,0.0
2,2,Bronchiolitis,0,0.0
3,3,COPD,1561,100.0
4,4,Healthy,0,0.0
5,5,LRTI,0,0.0
6,6,Pneumonia,0,0.0
7,7,URTI,0,0.0


Valid class IDs: True
Total predictions: 1561


In [54]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

all_class_ids = list(range(8))

def evaluate_saved_predictions(df, model_name):
    y_true = df["true_label"].to_numpy(dtype=int)
    y_pred = df["predicted_label"].to_numpy(dtype=int)

    print(f"\n{'=' * 65}")
    print(model_name)
    print("=" * 65)

    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.6f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.6f}")
    print(
        f"Macro F1:          "
        f"{f1_score(y_true, y_pred, labels=all_class_ids, average='macro', zero_division=0):.6f}"
    )
    print(
        f"Weighted F1:       "
        f"{f1_score(y_true, y_pred, labels=all_class_ids, average='weighted', zero_division=0):.6f}"
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=all_class_ids,
        target_names=expected_class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T
    display(
        report_df.loc[
            expected_class_names,
            ["precision", "recall", "f1-score", "support"]
        ].round(4)
    )

    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=all_class_ids),
        index=expected_class_names,
        columns=expected_class_names
    ))

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            labels=all_class_ids,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            labels=all_class_ids,
            average="weighted",
            zero_division=0
        )
    }


metrics_12a = evaluate_saved_predictions(
    multifeature_predictions_df,
    "Multi-feature CNN (12A)"
)

metrics_12b = evaluate_saved_predictions(
    regularized_predictions_df,
    "Regularized Multi-feature CNN (12B)"
)

multifeature_validation_metrics = pd.DataFrame(
    [metrics_12a, metrics_12b]
)

print("\nSummary comparison:")
display(multifeature_validation_metrics.round(4))



Multi-feature CNN (12A)
Accuracy:          0.855221
Balanced accuracy: 0.166417
Macro F1:          0.115245
Weighted F1:       0.789662


,precision,recall,f1-score,support
Asthma,0.0000,0.0000,0.000,0.0
Bronchiectasis,0.0000,0.0000,0.000,20.0
Bronchiolitis,0.0000,0.0000,0.000,19.0
COPD,0.8563,0.9985,0.922,1337.0
Healthy,0.0000,0.0000,0.000,84.0
LRTI,0.0000,0.0000,0.000,0.0
Pneumonia,0.0000,0.0000,0.000,52.0
URTI,0.0000,0.0000,0.000,49.0


Confusion matrix (rows=true, columns=predicted):


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,0,0,0,0,0
Bronchiectasis,0,0,0,20,0,0,0,0
Bronchiolitis,0,0,0,19,0,0,0,0
COPD,0,1,0,1335,0,0,1,0
Healthy,0,0,0,84,0,0,0,0
LRTI,0,0,0,0,0,0,0,0
Pneumonia,0,0,0,52,0,0,0,0
URTI,0,0,0,49,0,0,0,0



Regularized Multi-feature CNN (12B)
Accuracy:          0.856502
Balanced accuracy: 0.166667
Macro F1:          0.115338
Weighted F1:       0.790299


,precision,recall,f1-score,support
Asthma,0.0000,0.0,0.0000,0.0
Bronchiectasis,0.0000,0.0,0.0000,20.0
Bronchiolitis,0.0000,0.0,0.0000,19.0
COPD,0.8565,1.0,0.9227,1337.0
Healthy,0.0000,0.0,0.0000,84.0
LRTI,0.0000,0.0,0.0000,0.0
Pneumonia,0.0000,0.0,0.0000,52.0
URTI,0.0000,0.0,0.0000,49.0


Confusion matrix (rows=true, columns=predicted):


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,0,0,0,0,0
Bronchiectasis,0,0,0,20,0,0,0,0
Bronchiolitis,0,0,0,19,0,0,0,0
COPD,0,0,0,1337,0,0,0,0
Healthy,0,0,0,84,0,0,0,0
LRTI,0,0,0,0,0,0,0,0
Pneumonia,0,0,0,52,0,0,0,0
URTI,0,0,0,49,0,0,0,0



Summary comparison:


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Multi-feature CNN (12A),0.8552,0.1664,0.1152,0.7897
1,Regularized Multi-feature CNN (12B),0.8565,0.1667,0.1153,0.7903


In [55]:

expected_dl_artifacts = {
    "lightweight_mel_cnn": [
        "best_lightweight_mel_cnn.pt",
        "cnn_test_predictions.csv",
        "cnn_training_config.json",
        "cnn_training_history.csv",
        "logmel_train_normalization.csv",
    ],
    "multifeature_cnn": [
        "best_multifeature_cnn.pt",
        "multifeature_cnn_test_predictions.csv",
        "multifeature_cnn_training_config.json",
        "multifeature_train_normalization.csv",
    ],
    "regularized_multifeature_cnn": [
        "best_regularized_multifeature_cnn.pt",
        "test_predictions.csv",
        "training_config.json",
        "training_history.csv",
    ],
    "cnn_lstm": [
        "best_cnn_lstm.pt",
        "cnn_lstm_test_predictions.csv",
        "cnn_lstm_training_config.json",
        "cnn_lstm_training_history.csv",
        "logmel_train_normalization.csv",
    ],
}

artifact_records = []

for model_folder, filenames in expected_dl_artifacts.items():
    model_dir = DL_DIR / model_folder

    for filename in filenames:
        artifact_path = model_dir / filename

        artifact_records.append({
            "model_folder": model_folder,
            "artifact": filename,
            "exists": artifact_path.is_file(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.is_file()
                else None
            ),
        })

artifact_audit_df = pd.DataFrame(artifact_records)

display(artifact_audit_df)

missing_artifacts = artifact_audit_df.loc[
    ~artifact_audit_df["exists"]
]

print("\nExpected artifacts:", len(artifact_audit_df))
print("Present artifacts:", int(artifact_audit_df["exists"].sum()))
print("Missing artifacts:", len(missing_artifacts))

if len(missing_artifacts):
    display(missing_artifacts)
else:
    print("PASS: All listed deep-learning artifacts are present.")


,model_folder,artifact,exists,size_bytes
0,lightweight_mel_cnn,best_lightweight_mel_cnn.pt,True,310935
1,lightweight_mel_cnn,cnn_test_predictions.csv,True,223653
2,lightweight_mel_cnn,cnn_training_config.json,True,459
3,lightweight_mel_cnn,cnn_training_history.csv,True,3380
4,lightweight_mel_cnn,logmel_train_normalization.csv,True,62
5,multifeature_cnn,best_multifeature_cnn.pt,True,464538
6,multifeature_cnn,multifeature_cnn_test_predictions.csv,True,167848
7,multifeature_cnn,multifeature_cnn_training_config.json,True,528
8,multifeature_cnn,multifeature_train_normalization.csv,True,182
9,regularized_multifeature_cnn,best_regularized_multifeature_cnn.pt,True,465550



Expected artifacts: 18
Present artifacts: 18
Missing artifacts: 0
PASS: All listed deep-learning artifacts are present.


In [56]:

# Build a concise, auditable summary from results already verified.

validation_summary = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "100% agreement with saved predictions"
    },
    {
        "model": "SVM",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "100% agreement with saved predictions"
    },
    {
        "model": "Random Forest",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "100% agreement with saved predictions"
    },
    {
        "model": "Lightweight Mel CNN",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "1561/1561 predictions matched"
    },
    {
        "model": "CNN-LSTM",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "1561/1561 predictions matched"
    },
    {
        "model": "Multi-feature CNN (12A)",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "1561/1561 predictions matched"
    },
    {
        "model": "Regularized Multi-feature CNN (12B)",
        "validation_check": "Saved predictions reproduced",
        "result": "PASS",
        "details": "1561/1561 predictions matched"
    },
    {
        "model": "Multi-feature CNN (12A)",
        "validation_check": "Probability integrity and label consistency",
        "result": "PASS",
        "details": "Finite probabilities; row sums approximately 1; argmax matches labels"
    },
    {
        "model": "Regularized Multi-feature CNN (12B)",
        "validation_check": "Probability integrity and label consistency",
        "result": "PASS",
        "details": "Finite probabilities; row sums approximately 1; argmax matches labels"
    },
    {
        "model": "Deep-learning artifacts",
        "validation_check": "Expected artifact inventory",
        "result": "PASS",
        "details": "18/18 expected artifacts present"
    },
])

display(validation_summary)

# Add measured performance results for the two multi-feature models.
display(multifeature_validation_metrics.round(6))

# Save new validation outputs; do not overwrite model artifacts.
validation_output_dir = PROJECT_ROOT / "reports" / "validation"
validation_output_dir.mkdir(parents=True, exist_ok=True)

summary_path = validation_output_dir / "model_validation_summary.csv"
metrics_path = validation_output_dir / "multifeature_performance_metrics.csv"

validation_summary.to_csv(summary_path, index=False)
multifeature_validation_metrics.to_csv(metrics_path, index=False)

print("\nSaved validation summary:", summary_path)
print("Saved performance metrics:", metrics_path)
print("\nPASS: Validation summaries saved.")


,model,validation_check,result,details
0,Logistic Regression,Saved predictions reproduced,PASS,100% agreement with saved predictions
1,SVM,Saved predictions reproduced,PASS,100% agreement with saved predictions
2,Random Forest,Saved predictions reproduced,PASS,100% agreement with saved predictions
3,Lightweight Mel CNN,Saved predictions reproduced,PASS,1561/1561 predictions matched
4,CNN-LSTM,Saved predictions reproduced,PASS,1561/1561 predictions matched
5,Multi-feature CNN (12A),Saved predictions reproduced,PASS,1561/1561 predictions matched
6,Regularized Multi-feature CNN (12B),Saved predictions reproduced,PASS,1561/1561 predictions matched
7,Multi-feature CNN (12A),Probability integrity and label consistency,PASS,Finite probabilities; row sums approximately 1...
8,Regularized Multi-feature CNN (12B),Probability integrity and label consistency,PASS,Finite probabilities; row sums approximately 1...
9,Deep-learning artifacts,Expected artifact inventory,PASS,18/18 expected artifacts present


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Multi-feature CNN (12A),0.855221,0.166417,0.115245,0.789662
1,Regularized Multi-feature CNN (12B),0.856502,0.166667,0.115338,0.790299



Saved validation summary: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/validation/model_validation_summary.csv
Saved performance metrics: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/validation/multifeature_performance_metrics.csv

PASS: Validation summaries saved.


In [57]:

assert summary_path.is_file(), "Validation summary CSV is missing."
assert metrics_path.is_file(), "Performance metrics CSV is missing."

saved_summary_df = pd.read_csv(summary_path)
saved_metrics_df = pd.read_csv(metrics_path)

print("Validation summary exists:", summary_path.is_file())
print("Performance metrics exists:", metrics_path.is_file())

print("\nValidation summary rows:", len(saved_summary_df))
print("Performance metrics rows:", len(saved_metrics_df))

print("\nValidation status counts:")
print(saved_summary_df["result"].value_counts())

print("\nSaved performance metrics:")
display(saved_metrics_df)

assert len(saved_summary_df) == len(validation_summary)
assert len(saved_metrics_df) == 2
assert (saved_summary_df["result"] == "PASS").all()

print("\nPASS: Both CSV files were saved and read back successfully.")


Validation summary exists: True
Performance metrics exists: True

Validation summary rows: 10
Performance metrics rows: 2

Validation status counts:
result
PASS    10
Name: count, dtype: int64

Saved performance metrics:


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Multi-feature CNN (12A),0.855221,0.166417,0.115245,0.789662
1,Regularized Multi-feature CNN (12B),0.856502,0.166667,0.115338,0.790299



PASS: Both CSV files were saved and read back successfully.
